# 12 — Assembly101: Hugging Face Download and Verification

This notebook replaces the earlier Assembly101 conversion notebook.

The previous notebook failed because the local Assembly101 directory existed, but it did not contain the raw annotations or recordings. This notebook performs the missing setup step:

```text
Hugging Face access
    → download Assembly101 annotations safely
    → inspect and extract annotation archives
    → inventory recordings without downloading the full dataset
    → optionally download one fixed RGB view for a small smoke subset
    → save manifests for the next conversion notebook
```

## Safety

The complete Assembly101 repository is several terabytes. This notebook **does not download the whole repository**.

Default behavior:

```text
download annotations: yes
download metadata/README: yes
download TSM features: no
download DINOv2 features: no
download recordings: no
```

Optional smoke video behavior:

```text
one fixed RGB view: v1 / C10095_rgb.mp4
number of recordings: 1 by default
```

Before running, open the Assembly101 Hugging Face page in a browser, sign in, and accept the gated-dataset access conditions:

```text
https://huggingface.co/datasets/cvml-nus/assembly101
```

Use a personal Hugging Face **read token**. Do not place the token directly in notebook source code.

## 1. Mount Google Drive

In [21]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


## 2. Install dependencies

In [22]:
%pip install -q -U "huggingface_hub[hf_xet]" pandas tqdm

## 3. Imports and configuration

In [23]:
from pathlib import Path
import os
import re
import json
import csv
import shutil
import tarfile
import zipfile
import hashlib
from collections import Counter, defaultdict

import pandas as pd
from tqdm.auto import tqdm

from huggingface_hub import (
    HfApi,
    hf_hub_download,
    snapshot_download,
    login,
    notebook_login,
    get_token,
)
from huggingface_hub.errors import (
    GatedRepoError,
    RepositoryNotFoundError,
    HfHubHTTPError,
)

REPO_ID = "cvml-nus/assembly101"
REPO_TYPE = "dataset"
REVISION = "main"

DRIVE_ROOT = Path("/content/drive/MyDrive/mmf_tas_lab_data")
ASSEMBLY_ROOT = DRIVE_ROOT / "assembly101"
MANIFEST_ROOT = ASSEMBLY_ROOT / "manifests"

# Safe default: only annotations and small metadata.
DOWNLOAD_ANNOTATIONS = True
DOWNLOAD_SKILL_LABELS = True

# Do not enable these before reviewing storage requirements.
DOWNLOAD_TSM_FEATURES = False
DOWNLOAD_DINOV2_FEATURES = False

# Optional smoke test using raw video.
DOWNLOAD_SMOKE_VIDEOS = False
SMOKE_VIEW = "v1"
MAX_SMOKE_RECORDINGS = 1

# If None, choose the first sorted recording with the requested camera file.
# You can set exact recording folder names after inspecting recording_candidates.csv.
SMOKE_RECORDING_NAMES = None

# Extract small annotation archives after download.
AUTO_EXTRACT_ANNOTATION_ARCHIVES = True

# Download concurrency. Smaller is safer for Drive.
MAX_WORKERS = 4

VIEW_TO_CAMERA = {
    "v1": "C10095_rgb.mp4",
    "v2": "C10115_rgb.mp4",
    "v3": "C10118_rgb.mp4",
    "v4": "C10119_rgb.mp4",
    "v5": "C10379_rgb.mp4",
    "v6": "C10390_rgb.mp4",
    "v7": "C10395_rgb.mp4",
    "v8": "C10404_rgb.mp4",
    "e1": (
        "HMC_84346135_mono10bit.mp4",
        "HMC_21176875_mono10bit.mp4",
    ),
    "e2": (
        "HMC_84347414_mono10bit.mp4",
        "HMC_21176623_mono10bit.mp4",
    ),
    "e3": (
        "HMC_84355350_mono10bit.mp4",
        "HMC_21110305_mono10bit.mp4",
    ),
    "e4": (
        "HMC_84358933_mono10bit.mp4",
        "HMC_21179183_mono10bit.mp4",
    ),
}

if SMOKE_VIEW not in VIEW_TO_CAMERA:
    raise ValueError(f"Unknown SMOKE_VIEW={SMOKE_VIEW!r}. Choose one of {sorted(VIEW_TO_CAMERA)}")

ASSEMBLY_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

print("REPO_ID:", REPO_ID)
print("ASSEMBLY_ROOT:", ASSEMBLY_ROOT)
print("DOWNLOAD_ANNOTATIONS:", DOWNLOAD_ANNOTATIONS)
print("DOWNLOAD_SMOKE_VIDEOS:", DOWNLOAD_SMOKE_VIDEOS)
print("SMOKE_VIEW:", SMOKE_VIEW)
print("MAX_SMOKE_RECORDINGS:", MAX_SMOKE_RECORDINGS)

REPO_ID: cvml-nus/assembly101
ASSEMBLY_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/assembly101
DOWNLOAD_ANNOTATIONS: True
DOWNLOAD_SMOKE_VIDEOS: False
SMOKE_VIEW: v1
MAX_SMOKE_RECORDINGS: 1


## 4. Authenticate with Hugging Face

Recommended Colab setup:

1. Open **Colab → Secrets**.
2. Add a secret named `HF_TOKEN`.
3. Enable notebook access for that secret.
4. Use a Hugging Face personal token with read permission.

If `HF_TOKEN` is unavailable, the cell opens the Hugging Face notebook login UI.

In [24]:
def authenticate_huggingface():
    token = None

    # Prefer a Colab secret, without printing it.
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

    if token:
        login(token=token, add_to_git_credential=False)
        print("Authenticated using Colab secret HF_TOKEN.")
        return

    if get_token() is not None:
        print("A Hugging Face token is already available in this runtime.")
        return

    print("No HF_TOKEN secret found. Opening the notebook login widget.")
    notebook_login(skip_if_logged_in=True)

authenticate_huggingface()

assert get_token() is not None, (
    "Authentication did not complete. Add HF_TOKEN in Colab Secrets "
    "or rerun this cell and finish the notebook login."
)

Authenticated using Colab secret HF_TOKEN.


## 5. Verify gated-dataset access

In [25]:
api = HfApi()

try:
    who = api.whoami(token=True)
    print("Authenticated Hugging Face user:", who.get("name", "<unknown>"))

    dataset_info = api.dataset_info(
        repo_id=REPO_ID,
        revision=REVISION,
        files_metadata=True,
        token=True,
    )

    print("Dataset access: OK")
    print("Repository:", dataset_info.id)
    print("Revision SHA:", dataset_info.sha)
    print("Private:", dataset_info.private)
    print("Gated:", dataset_info.gated)

except GatedRepoError as e:
    raise RuntimeError(
        "The Hugging Face account is authenticated but does not have access to "
        "Assembly101. Open https://huggingface.co/datasets/cvml-nus/assembly101, "
        "accept the access conditions, then rerun this cell."
    ) from e

except RepositoryNotFoundError as e:
    raise RuntimeError(
        "Assembly101 could not be accessed. Check REPO_ID and authentication."
    ) from e

except HfHubHTTPError as e:
    raise RuntimeError(
        f"Hugging Face access check failed: {e}. "
        "Check the token, gated access, and internet connection."
    ) from e

Authenticated Hugging Face user: Bonart
Dataset access: OK
Repository: cvml-nus/assembly101
Revision SHA: bfc15ea5e3f0bc8f8c232af6c1b45aa137a9d967
Private: False
Gated: auto


## 6. List the remote repository without downloading it

In [26]:
repo_files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    revision=REVISION,
    token=True,
)

print("Remote files:", len(repo_files))

top_level_counts = Counter()
for path in repo_files:
    top_level = path.split("/", 1)[0]
    top_level_counts[top_level] += 1

display(
    pd.DataFrame(
        [{"top_level": k, "num_files": v} for k, v in sorted(top_level_counts.items())]
    )
)

print("\nFirst 100 remote paths:")
for path in repo_files[:100]:
    print(path)

repo_inventory_path = MANIFEST_ROOT / "hf_repo_file_inventory.txt"
repo_inventory_path.write_text("\n".join(repo_files) + "\n", encoding="utf-8")
print("\nSaved:", repo_inventory_path)

Remote files: 5392


,top_level,num_files
0,.gitattributes,1
1,AssemblyPoses.zip,1
2,DINOv2_features,1
3,README.md,1
4,TSM_features,17
5,annotations,695
6,poses@60fps,354
7,recordings,4321
8,skill_labels.zip,1



First 100 remote paths:
.gitattributes
AssemblyPoses.zip
DINOv2_features/C10119_rgb.zip
README.md
TSM_features/C10095_rgb.zip
TSM_features/C10115_rgb.zip
TSM_features/C10118_rgb.zip
TSM_features/C10119_rgb.zip
TSM_features/C10379_rgb.zip
TSM_features/C10390_rgb.zip
TSM_features/C10395_rgb.zip
TSM_features/C10404_rgb.zip
TSM_features/HMC_21110305_mono10bit.zip
TSM_features/HMC_21176623_mono10bit.zip
TSM_features/HMC_21176875_mono10bit.zip
TSM_features/HMC_21179183_mono10bit.zip
TSM_features/HMC_84346135_mono10bit.zip
TSM_features/HMC_84347414_mono10bit.zip
TSM_features/HMC_84355350_mono10bit.zip
TSM_features/HMC_84358933_mono10bit.zip
TSM_features/read_lmdb.py
annotations/README.md
annotations/coarse-annotations/actions.csv
annotations/coarse-annotations/coarse_labels/assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.txt
annotations/coarse-annotations/coarse_labels/assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253.txt
annotations/coarse-anno

## 7. Build a recording/view inventory

In [27]:
recording_rows = []

for remote_path in repo_files:
    parts = remote_path.split("/")
    if len(parts) != 3 or parts[0] != "recordings":
        continue

    _, recording_name, filename = parts

    view_name = None
    for candidate_view, camera_names in VIEW_TO_CAMERA.items():
        if isinstance(camera_names, str):
            camera_names = (camera_names,)
        if filename in camera_names:
            view_name = candidate_view
            break

    recording_rows.append({
        "recording_name": recording_name,
        "filename": filename,
        "remote_path": remote_path,
        "view": view_name,
    })

recording_inventory = pd.DataFrame(recording_rows)

if recording_inventory.empty:
    raise RuntimeError("No recording files were found in the remote repository inventory.")

print("Recording files:", len(recording_inventory))
print("Unique recording folders:", recording_inventory["recording_name"].nunique())
print("Recognized views:", recording_inventory["view"].notna().sum())

display(
    recording_inventory["view"]
    .fillna("unrecognized")
    .value_counts()
    .rename_axis("view")
    .reset_index(name="num_files")
)

recording_inventory_path = MANIFEST_ROOT / "recording_file_inventory.csv"
recording_inventory.to_csv(recording_inventory_path, index=False)

camera_names = VIEW_TO_CAMERA[SMOKE_VIEW]
if isinstance(camera_names, str):
    camera_names = (camera_names,)

smoke_candidates = (
    recording_inventory[
        recording_inventory["filename"].isin(camera_names)
    ]
    .sort_values(["recording_name", "filename"])
    .reset_index(drop=True)
)

smoke_candidates_path = MANIFEST_ROOT / f"recording_candidates_{SMOKE_VIEW}.csv"
smoke_candidates.to_csv(smoke_candidates_path, index=False)

print("Candidates for", SMOKE_VIEW, ":", len(smoke_candidates))
print("Saved:", recording_inventory_path)
print("Saved:", smoke_candidates_path)
display(smoke_candidates.head(20))

Recording files: 4321
Unique recording folders: 362
Recognized views: 4321


,view,num_files
0,v1,362
1,v2,362
2,v3,362
3,v4,362
4,v5,362
5,v6,362
6,v7,362
7,v8,362
8,e2,357
9,e4,357


Candidates for v1 : 362
Saved: /content/drive/MyDrive/mmf_tas_lab_data/assembly101/manifests/recording_file_inventory.csv
Saved: /content/drive/MyDrive/mmf_tas_lab_data/assembly101/manifests/recording_candidates_v1.csv


,recording_name,filename,remote_path,view
0,nusar-2021_action_both_9011-a01_9011_user_id_2...,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-a01_901...,v1
1,nusar-2021_action_both_9011-b06b_9011_user_id_...,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-b06b_90...,v1
2,nusar-2021_action_both_9011-b08c_9011_user_id_...,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-b08c_90...,v1
3,nusar-2021_action_both_9011-c01c_9011_user_id_...,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-c01c_90...,v1
4,nusar-2021_action_both_9011-c03f_9011_user_id_...,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-c03f_90...,v1
5,nusar-2021_action_both_9011-c13b_9011_user_id_...,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-c13b_90...,v1
6,nusar-2021_action_both_9012-a16_9012_user_id_2...,C10095_rgb.mp4,recordings/nusar-2021_action_both_9012-a16_901...,v1
7,nusar-2021_action_both_9012-a17_9012_user_id_2...,C10095_rgb.mp4,recordings/nusar-2021_action_both_9012-a17_901...,v1
8,nusar-2021_action_both_9012-b06d_9012_user_id_...,C10095_rgb.mp4,recordings/nusar-2021_action_both_9012-b06d_90...,v1
9,nusar-2021_action_both_9012-c06d_9012_user_id_...,C10095_rgb.mp4,recordings/nusar-2021_action_both_9012-c06d_90...,v1


## 8. Preview the planned download

In [28]:
allow_patterns = ["README.md"]

if DOWNLOAD_ANNOTATIONS:
    allow_patterns.append("annotations/**")

if DOWNLOAD_SKILL_LABELS:
    allow_patterns.append("skill_labels.zip")

if DOWNLOAD_TSM_FEATURES:
    allow_patterns.append("TSM_features/**")

if DOWNLOAD_DINOV2_FEATURES:
    allow_patterns.append("DINOv2_features/**")

print("Planned snapshot patterns:")
for pattern in allow_patterns:
    print(" -", pattern)

print("\nImportant:")
print("recordings/** is deliberately NOT included in snapshot_download.")
print("Smoke videos, if enabled, are downloaded one file at a time later.")

dry_run_rows = []

try:
    dry_run_info = snapshot_download(
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        revision=REVISION,
        local_dir=ASSEMBLY_ROOT,
        allow_patterns=allow_patterns,
        token=True,
        max_workers=MAX_WORKERS,
        dry_run=True,
    )

    for item in dry_run_info:
        dry_run_rows.append({
            "filename": getattr(item, "filename", None),
            "file_size_bytes": getattr(item, "file_size", None),
            "will_download": getattr(item, "will_download", None),
            "is_cached": getattr(item, "is_cached", None),
        })

    dry_run_df = pd.DataFrame(dry_run_rows)

    if not dry_run_df.empty:
        total_bytes = pd.to_numeric(
            dry_run_df["file_size_bytes"], errors="coerce"
        ).fillna(0).sum()

        download_bytes = pd.to_numeric(
            dry_run_df.loc[
                dry_run_df["will_download"].fillna(True),
                "file_size_bytes",
            ],
            errors="coerce",
        ).fillna(0).sum()

        print(f"\nMatched remote files: {len(dry_run_df)}")
        print(f"Total matched size: {total_bytes / 1024**3:.3f} GiB")
        print(f"Expected new download: {download_bytes / 1024**3:.3f} GiB")
        display(dry_run_df.head(100))

        dry_run_path = MANIFEST_ROOT / "planned_snapshot_download.csv"
        dry_run_df.to_csv(dry_run_path, index=False)
        print("Saved:", dry_run_path)

except TypeError:
    # Compatibility fallback if dry_run is unavailable in an older environment.
    print(
        "This huggingface_hub build does not expose dry_run. "
        "The actual download still remains restricted by allow_patterns."
    )

Planned snapshot patterns:
 - README.md
 - annotations/**
 - skill_labels.zip

Important:
recordings/** is deliberately NOT included in snapshot_download.
Smoke videos, if enabled, are downloaded one file at a time later.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

[dry-run] Fetching 697 files:   0%|          | 0/697 [00:00<?, ?it/s]


Matched remote files: 697
Total matched size: 0.172 GiB
Expected new download: 0.172 GiB


,filename,file_size_bytes,will_download,is_cached
0,README.md,4305,True,False
1,annotations/README.md,7852,True,False
2,annotations/coarse-annotations/actions.csv,9085,True,False
3,annotations/coarse-annotations/coarse_labels/a...,514,True,False
4,annotations/coarse-annotations/coarse_labels/a...,323,True,False
...,...,...,...,...
95,annotations/coarse-annotations/coarse_labels/a...,454,True,False
96,annotations/coarse-annotations/coarse_labels/a...,619,True,False
97,annotations/coarse-annotations/coarse_labels/a...,410,True,False
98,annotations/coarse-annotations/coarse_labels/a...,415,True,False


Saved: /content/drive/MyDrive/mmf_tas_lab_data/assembly101/manifests/planned_snapshot_download.csv


## 9. Download annotations and selected small metadata

This uses `allow_patterns`. It does not download `recordings`, TSM features, DINOv2 features, or poses unless explicitly enabled in the configuration.

In [29]:
snapshot_path = snapshot_download(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    revision=REVISION,
    local_dir=ASSEMBLY_ROOT,
    allow_patterns=allow_patterns,
    token=True,
    max_workers=MAX_WORKERS,
)

print("Snapshot materialized under:", snapshot_path)
print("Local Assembly101 root:", ASSEMBLY_ROOT)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 697 files:   0%|          | 0/697 [00:00<?, ?it/s]

Snapshot materialized under: /content/drive/MyDrive/mmf_tas_lab_data/assembly101
Local Assembly101 root: /content/drive/MyDrive/mmf_tas_lab_data/assembly101


## 10. Safely extract annotation archives

In [30]:
def is_within_directory(directory: Path, target: Path) -> bool:
    directory = directory.resolve()
    target = target.resolve()
    return os.path.commonpath([str(directory), str(target)]) == str(directory)

def safe_extract_zip(archive_path: Path, destination: Path):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as zf:
        for member in zf.infolist():
            target = destination / member.filename
            if not is_within_directory(destination, target):
                raise RuntimeError(f"Unsafe ZIP member path: {member.filename}")
        zf.extractall(destination)

def safe_extract_tar(archive_path: Path, destination: Path):
    destination.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path) as tf:
        for member in tf.getmembers():
            target = destination / member.name
            if not is_within_directory(destination, target):
                raise RuntimeError(f"Unsafe TAR member path: {member.name}")
        tf.extractall(destination)

annotation_root = ASSEMBLY_ROOT / "annotations"
archive_rows = []

if AUTO_EXTRACT_ANNOTATION_ARCHIVES and annotation_root.exists():
    archives = sorted(
        [
            p for p in annotation_root.rglob("*")
            if p.is_file() and (
                p.suffix.lower() == ".zip"
                or p.name.lower().endswith((".tar.gz", ".tgz", ".tar"))
            )
        ]
    )

    for archive in archives:
        if archive.suffix.lower() == ".zip":
            destination = archive.parent / archive.stem
            extractor = "zip"
        else:
            name = archive.name
            for suffix in [".tar.gz", ".tgz", ".tar"]:
                if name.lower().endswith(suffix):
                    name = name[:-len(suffix)]
                    break
            destination = archive.parent / name
            extractor = "tar"

        before_exists = destination.exists() and any(destination.iterdir())

        if before_exists:
            status = "already_extracted"
        else:
            print("Extracting:", archive)
            if extractor == "zip":
                safe_extract_zip(archive, destination)
            else:
                safe_extract_tar(archive, destination)
            status = "extracted"

        archive_rows.append({
            "archive": str(archive),
            "destination": str(destination),
            "status": status,
            "size_mb": round(archive.stat().st_size / 1024**2, 3),
        })

archive_df = pd.DataFrame(archive_rows)
display(archive_df)

archive_manifest_path = MANIFEST_ROOT / "annotation_archive_extraction.csv"
archive_df.to_csv(archive_manifest_path, index=False)
print("Saved:", archive_manifest_path)

""


Saved: /content/drive/MyDrive/mmf_tas_lab_data/assembly101/manifests/annotation_archive_extraction.csv


## 11. Inspect the downloaded annotation structure

In [31]:
def print_tree(root: Path, max_depth=4, max_entries_per_dir=80):
    root = Path(root)
    print(root)

    if not root.exists():
        print("  [missing]")
        return

    for current, dirs, files in os.walk(root):
        current_path = Path(current)
        depth = len(current_path.relative_to(root).parts)

        if depth >= max_depth:
            dirs[:] = []
            continue

        dirs[:] = sorted(dirs)[:max_entries_per_dir]
        files = sorted(files)[:max_entries_per_dir]
        indent = "  " * depth

        for d in dirs:
            print(f"{indent}📁 {d}/")
        for f in files:
            print(f"{indent}📄 {f}")

print_tree(annotation_root, max_depth=5)

/content/drive/MyDrive/mmf_tas_lab_data/assembly101/annotations
📁 coarse-annotations/
📁 fine-grained-annotations/
📄 README.md
  📁 coarse_labels/
  📁 coarse_splits/
  📄 actions.csv
  📄 coarse_seq_views.txt
  📄 tail_actions.txt
    📄 assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.txt
    📄 assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253.txt
    📄 assembly_nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736.txt
    📄 assembly_nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620.txt
    📄 assembly_nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239.txt
    📄 assembly_nusar-2021_action_both_9011-c13b_9011_user_id_2021-02-01_160915.txt
    📄 assembly_nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904.txt
    📄 assembly_nusar-2021_action_both_9012-a17_9012_user_id_2021-02-01_162209.txt
    📄 assembly_nusar-2021_action_both_9012-b06d_9012_user_id_2021-02-01_163713.txt
    📄 assembly_nusar-2021_acti

## 12. Create a local annotation inventory

In [32]:
ANNOTATION_SUFFIXES = {
    ".csv", ".tsv", ".txt", ".json", ".jsonl",
    ".pkl", ".pickle", ".parquet", ".npy", ".npz",
    ".zip", ".gz", ".tar",
}

annotation_files = []

if annotation_root.exists():
    for path in sorted(annotation_root.rglob("*")):
        if not path.is_file():
            continue

        annotation_files.append({
            "relative_path": str(path.relative_to(ASSEMBLY_ROOT)),
            "absolute_path": str(path),
            "suffix": path.suffix.lower(),
            "size_bytes": path.stat().st_size,
            "size_mb": round(path.stat().st_size / 1024**2, 4),
            "likely_annotation_table": (
                path.suffix.lower() in ANNOTATION_SUFFIXES
                or any(
                    key in path.name.lower()
                    for key in [
                        "annot", "action", "segment", "label",
                        "split", "train", "test", "validation",
                    ]
                )
            ),
        })

annotation_inventory = pd.DataFrame(annotation_files)

print("Local annotation files:", len(annotation_inventory))
if not annotation_inventory.empty:
    print(
        "Total annotation size:",
        round(annotation_inventory["size_bytes"].sum() / 1024**2, 3),
        "MiB",
    )
    display(
        annotation_inventory.sort_values(
            ["likely_annotation_table", "size_bytes"],
            ascending=[False, False],
        ).head(200)
    )

annotation_inventory_path = MANIFEST_ROOT / "annotation_file_inventory.csv"
annotation_inventory.to_csv(annotation_inventory_path, index=False)
print("Saved:", annotation_inventory_path)

if annotation_inventory.empty:
    raise RuntimeError(
        "No local annotation files were found after the download. "
        "Check gated access and the remote inventory."
    )

Local annotation files: 695
Total annotation size: 175.907 MiB


,relative_path,absolute_path,suffix,size_bytes,size_mb,likely_annotation_table
693,annotations/fine-grained-annotations/train.csv,/content/drive/MyDrive/mmf_tas_lab_data/assemb...,.csv,100677017,96.0131,True
692,annotations/fine-grained-annotations/test.csv,/content/drive/MyDrive/mmf_tas_lab_data/assemb...,.csv,47993475,45.7701,True
694,annotations/fine-grained-annotations/validatio...,/content/drive/MyDrive/mmf_tas_lab_data/assemb...,.csv,34544546,32.9442,True
682,annotations/coarse-annotations/coarse_seq_view...,/content/drive/MyDrive/mmf_tas_lab_data/assemb...,.txt,751869,0.7170,True
690,annotations/fine-grained-annotations/actions.csv,/content/drive/MyDrive/mmf_tas_lab_data/assemb...,.csv,81530,0.0778,True
...,...,...,...,...,...,...
333,annotations/coarse-annotations/coarse_labels/a...,/content/drive/MyDrive/mmf_tas_lab_data/assemb...,.txt,512,0.0005,True
65,annotations/coarse-annotations/coarse_labels/a...,/content/drive/MyDrive/mmf_tas_lab_data/assemb...,.txt,511,0.0005,True
310,annotations/coarse-annotations/coarse_labels/a...,/content/drive/MyDrive/mmf_tas_lab_data/assemb...,.txt,510,0.0005,True
132,annotations/coarse-annotations/coarse_labels/a...,/content/drive/MyDrive/mmf_tas_lab_data/assemb...,.txt,509,0.0005,True


Saved: /content/drive/MyDrive/mmf_tas_lab_data/assembly101/manifests/annotation_file_inventory.csv


## 13. Optional: download one fixed RGB view for a smoke subset

Default is disabled.

To enable it, return to the configuration cell and set:

```python
DOWNLOAD_SMOKE_VIDEOS = True
SMOKE_VIEW = "v1"
MAX_SMOKE_RECORDINGS = 1
```

`v1` corresponds to `C10095_rgb.mp4`.

The notebook downloads exact files one by one. It never uses `recordings/**`.

In [33]:
smoke_download_rows = []

if not DOWNLOAD_SMOKE_VIDEOS:
    print("Smoke video download is disabled. No recordings were downloaded.")

else:
    candidates = smoke_candidates.copy()

    if SMOKE_RECORDING_NAMES is not None:
        requested = set(SMOKE_RECORDING_NAMES)
        candidates = candidates[
            candidates["recording_name"].isin(requested)
        ].copy()

        missing_requested = sorted(
            requested - set(candidates["recording_name"])
        )
        if missing_requested:
            print("Requested recording names not found for", SMOKE_VIEW)
            for name in missing_requested:
                print(" -", name)

    candidates = candidates.head(MAX_SMOKE_RECORDINGS)

    if candidates.empty:
        raise RuntimeError(
            f"No remote recording candidates were found for view {SMOKE_VIEW}."
        )

    print("Downloading", len(candidates), "smoke video file(s).")

    for _, row in tqdm(
        candidates.iterrows(),
        total=len(candidates),
        desc="Smoke videos",
    ):
        remote_path = row["remote_path"]

        try:
            local_path = hf_hub_download(
                repo_id=REPO_ID,
                filename=remote_path,
                repo_type=REPO_TYPE,
                revision=REVISION,
                local_dir=ASSEMBLY_ROOT,
                token=True,
            )

            local_path = Path(local_path)
            smoke_download_rows.append({
                "recording_name": row["recording_name"],
                "view": SMOKE_VIEW,
                "remote_path": remote_path,
                "local_path": str(local_path),
                "exists": local_path.exists(),
                "size_gb": (
                    round(local_path.stat().st_size / 1024**3, 3)
                    if local_path.exists()
                    else None
                ),
                "status": "downloaded",
                "error": "",
            })

        except Exception as e:
            smoke_download_rows.append({
                "recording_name": row["recording_name"],
                "view": SMOKE_VIEW,
                "remote_path": remote_path,
                "local_path": "",
                "exists": False,
                "size_gb": None,
                "status": "failed",
                "error": repr(e),
            })

smoke_download_df = pd.DataFrame(smoke_download_rows)
display(smoke_download_df)

smoke_manifest_path = MANIFEST_ROOT / "smoke_video_download_manifest.csv"
smoke_download_df.to_csv(smoke_manifest_path, index=False)
print("Saved:", smoke_manifest_path)

Smoke video download is disabled. No recordings were downloaded.


""


Saved: /content/drive/MyDrive/mmf_tas_lab_data/assembly101/manifests/smoke_video_download_manifest.csv


## 14. Verify downloaded video files

In [34]:
video_extensions = {".mp4", ".avi", ".mkv", ".webm", ".mov"}

local_videos = [
    p for p in ASSEMBLY_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in video_extensions
]

video_rows = []

for path in sorted(local_videos):
    video_rows.append({
        "relative_path": str(path.relative_to(ASSEMBLY_ROOT)),
        "absolute_path": str(path),
        "size_gb": round(path.stat().st_size / 1024**3, 4),
        "nonempty": path.stat().st_size > 0,
    })

local_video_inventory = pd.DataFrame(video_rows)

print("Local video files:", len(local_video_inventory))
display(local_video_inventory.head(100))

local_video_inventory_path = MANIFEST_ROOT / "local_video_inventory.csv"
local_video_inventory.to_csv(local_video_inventory_path, index=False)
print("Saved:", local_video_inventory_path)

Local video files: 0


""


Saved: /content/drive/MyDrive/mmf_tas_lab_data/assembly101/manifests/local_video_inventory.csv


## 15. Optional FFprobe validation for local smoke videos

In [35]:
import subprocess

ffprobe_rows = []

for path in local_videos:
    cmd = [
        "ffprobe",
        "-v", "error",
        "-select_streams", "v:0",
        "-show_entries",
        "stream=codec_name,width,height,r_frame_rate,avg_frame_rate,nb_frames,duration",
        "-of", "json",
        str(path),
    ]

    try:
        proc = subprocess.run(
            cmd,
            text=True,
            capture_output=True,
            check=False,
        )

        payload = json.loads(proc.stdout) if proc.stdout.strip() else {}
        streams = payload.get("streams", [])
        stream = streams[0] if streams else {}

        ffprobe_rows.append({
            "path": str(path),
            "returncode": proc.returncode,
            "codec": stream.get("codec_name"),
            "width": stream.get("width"),
            "height": stream.get("height"),
            "r_frame_rate": stream.get("r_frame_rate"),
            "avg_frame_rate": stream.get("avg_frame_rate"),
            "nb_frames": stream.get("nb_frames"),
            "duration": stream.get("duration"),
            "stderr": proc.stderr[-1000:],
        })

    except Exception as e:
        ffprobe_rows.append({
            "path": str(path),
            "returncode": -1,
            "codec": None,
            "width": None,
            "height": None,
            "r_frame_rate": None,
            "avg_frame_rate": None,
            "nb_frames": None,
            "duration": None,
            "stderr": repr(e),
        })

ffprobe_df = pd.DataFrame(ffprobe_rows)
display(ffprobe_df)

ffprobe_manifest_path = MANIFEST_ROOT / "local_video_ffprobe.csv"
ffprobe_df.to_csv(ffprobe_manifest_path, index=False)
print("Saved:", ffprobe_manifest_path)

if local_videos:
    assert (ffprobe_df["returncode"] == 0).all(), (
        "At least one local video failed FFprobe validation."
    )
else:
    print("No local recordings to validate. This is expected in annotations-only mode.")

""


Saved: /content/drive/MyDrive/mmf_tas_lab_data/assembly101/manifests/local_video_ffprobe.csv
No local recordings to validate. This is expected in annotations-only mode.


## 16. Final validation and summary

In [36]:
annotation_file_count = int(len(annotation_inventory))
recording_folder_count_remote = int(
    recording_inventory["recording_name"].nunique()
)
local_video_count = int(len(local_video_inventory))

summary = {
    "status": "completed",
    "dataset": "Assembly101",
    "source": "Hugging Face dataset repository",
    "repo_id": REPO_ID,
    "revision": REVISION,
    "assembly_root": str(ASSEMBLY_ROOT),
    "download_configuration": {
        "download_annotations": DOWNLOAD_ANNOTATIONS,
        "download_skill_labels": DOWNLOAD_SKILL_LABELS,
        "download_tsm_features": DOWNLOAD_TSM_FEATURES,
        "download_dinov2_features": DOWNLOAD_DINOV2_FEATURES,
        "download_smoke_videos": DOWNLOAD_SMOKE_VIDEOS,
        "smoke_view": SMOKE_VIEW,
        "max_smoke_recordings": MAX_SMOKE_RECORDINGS,
    },
    "remote_inventory": {
        "num_repo_files": int(len(repo_files)),
        "num_recording_files": int(len(recording_inventory)),
        "num_recording_folders": recording_folder_count_remote,
        "num_candidates_for_smoke_view": int(len(smoke_candidates)),
    },
    "local_inventory": {
        "num_annotation_files": annotation_file_count,
        "num_local_video_files": local_video_count,
        "annotations_root_exists": annotation_root.exists(),
    },
    "paths": {
        "annotations_root": str(annotation_root),
        "manifests_root": str(MANIFEST_ROOT),
        "repo_inventory": str(repo_inventory_path),
        "recording_inventory": str(recording_inventory_path),
        "smoke_candidates": str(smoke_candidates_path),
        "annotation_inventory": str(annotation_inventory_path),
        "local_video_inventory": str(local_video_inventory_path),
        "ffprobe_manifest": str(ffprobe_manifest_path),
    },
    "next_step": (
        "Inspect the Assembly101 annotation format and convert temporal "
        "segmentation labels/splits to the MS-TCN-compatible representation."
    ),
}

summary_path = MANIFEST_ROOT / "assembly101_download_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2))
print("\nSaved:", summary_path)

assert annotation_root.exists(), "Annotation root is missing."
assert annotation_file_count > 0, "No annotation files were downloaded."

if DOWNLOAD_SMOKE_VIDEOS:
    assert local_video_count > 0, (
        "Smoke video download was enabled, but no local video was found."
    )

print("\nAssembly101 setup is ready for the next notebook.")
print("Next notebook:")
print("13_assembly101_annotations_to_mstcn_COLAB.ipynb")

{
  "status": "completed",
  "dataset": "Assembly101",
  "source": "Hugging Face dataset repository",
  "repo_id": "cvml-nus/assembly101",
  "revision": "main",
  "assembly_root": "/content/drive/MyDrive/mmf_tas_lab_data/assembly101",
  "download_configuration": {
    "download_annotations": true,
    "download_skill_labels": true,
    "download_tsm_features": false,
    "download_dinov2_features": false,
    "download_smoke_videos": false,
    "smoke_view": "v1",
    "max_smoke_recordings": 1
  },
  "remote_inventory": {
    "num_repo_files": 5392,
    "num_recording_files": 4321,
    "num_recording_folders": 362,
    "num_candidates_for_smoke_view": 362
  },
  "local_inventory": {
    "num_annotation_files": 695,
    "num_local_video_files": 0,
    "annotations_root_exists": true
  },
  "paths": {
    "annotations_root": "/content/drive/MyDrive/mmf_tas_lab_data/assembly101/annotations",
    "manifests_root": "/content/drive/MyDrive/mmf_tas_lab_data/assembly101/manifests",
    "repo_i

## Expected successful output

In default annotations-only mode:

```text
Dataset access: OK
Remote files: > 0
Unique recording folders: > 0
Local annotation files: > 0
Local video files: 0
status: completed
```

With smoke video mode enabled:

```text
Local video files: 1
FFprobe returncode: 0
status: completed
```